---
title: "最短路径——Bellman-Ford & SPFA 算法"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [6]:
from typing import List
from collections import deque
import math

## 📝 题目 787：K 站中转内最便宜的航班

::: {.callout-caution}
### 📖 题目描述
有 `n` 个城市通过一些航班连接。给你一个数组 `flights` ，其中 `flights[i] = `[$from_i$, $to_i$, $price_i$] ，表示该航班都从城市 $from_i$ 开始，以价格 $price_i$ 抵达 $to_i$。

现在给定所有的城市和航班，以及出发城市 `src` 和目的地 `dst`，你的任务是找到出一条最多经过 `k` 站中转的路线，使得从 `src` 到 `dst` 的 **价格最便宜** ，并返回该价格。 如果不存在这样的路线，则输出 `-1`。

**注意**：经过 `k` 站中转，意味着你从 `src` 到 `dst` 最多只能搭乘 `k + 1` 趟航班（走 `k + 1` 条边）。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 3`, `flights = [[0,1,100],[1,2,100],[0,2,500]]`, `src = 0`, `dst = 2`, `k = 1`
    * **输出**：`200`
    * **解释**：
      从 0 到 2 有两条航线：
      航线 1：0 -> 2 (花费 500，中转 0 次)
      航线 2：0 -> 1 -> 2 (花费 200，中转 1 次)
      最多允许中转 1 次，因此我们选择航线 2，花费 200。

* **示例 2**：
    * **输入**：`n = 3`, `flights = [[0,1,100],[1,2,100],[0,2,500]]`, `src = 0`, `dst = 2`, `k = 0`
    * **输出**：`500`
    * **解释**：
      题目同上，但最多允许中转 0 次（只能直飞）。所以只能选花费 500 的航线 1。
:::

In [26]:
class Solution787:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        dist = [math.inf] * n  # 初始化距离表，大小为 n (城市编号是 0 到 n-1)
        dist[src] = 0
        for _ in range(k + 1):  # 严格执行 k + 1 次遍历 (最多飞 k + 1 次)
            clone_dist = list(dist)  # 克隆历史状态，防止在同一次循环里发生“连环转机”
            for start, end, price in flights:
                clone_dist[end] = min(clone_dist[end], dist[start] + price)
            dist = clone_dist  # 本次循环结束，新状态正式上位
        return dist[dst] if dist[dst] != math.inf else -1

In [27]:
#| code-fold: true
def test_787(func):
    test_cases = [
        {"desc": "示例 1: 允许1次转机选便宜", "n": 3, "flights": [[0,1,100],[1,2,100],[0,2,500]], "src": 0, "dst": 2, "k": 1, "expected": 200},
        {"desc": "示例 2: 只能直飞选贵的", "n": 3, "flights": [[0,1,100],[1,2,100],[0,2,500]], "src": 0, "dst": 2, "k": 0, "expected": 500},
        {"desc": "找不到路线", "n": 3, "flights": [[0,1,100]], "src": 0, "dst": 2, "k": 0, "expected": -1},
        {"desc": "K足够大，选最便宜的迂回路线", "n": 4, "flights": [[0,1,1],[1,2,1],[2,3,1],[0,3,10]], "src": 0, "dst": 3, "k": 2, "expected": 3},
        {"desc": "K不够，只能选中等价位路线", "n": 4, "flights": [[0,1,1],[1,2,1],[2,3,1],[0,2,5],[2,3,1],[0,3,10]], "src": 0, "dst": 3, "k": 1, "expected": 6}, # 0->2->3 花费 5+1=6，转机1次
        {"desc": "包含环，测试防重拦截机制", "n": 3, "flights": [[0,1,100],[1,0,10],[1,2,100]], "src": 0, "dst": 2, "k": 1, "expected": 200},
        {"desc": "起点和终点根本不通", "n": 4, "flights": [[0,1,100],[2,3,100]], "src": 0, "dst": 3, "k": 2, "expected": -1},
        {"desc": "有多条同等转机次数的平行航线", "n": 3, "flights": [[0,1,100],[0,1,50],[1,2,100]], "src": 0, "dst": 2, "k": 1, "expected": 150},
        {"desc": "复杂的梯子网络", "n": 5, "flights": [[0,1,10],[1,2,10],[2,3,10],[3,4,10],[0,2,30],[2,4,30],[0,4,100]], "src": 0, "dst": 4, "k": 2, "expected": 50},
        {"desc": "极限: 单个节点不需要飞", "n": 1, "flights": [], "src": 0, "dst": 0, "k": 0, "expected": 0},
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["n"], tc["flights"], tc["src"], tc["dst"], tc["k"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_787(Solution787().findCheapestPrice)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 200  | 200  | 示例 1: 允许1次转机选便宜
2    | ✅ PASS | 500  | 500  | 示例 2: 只能直飞选贵的
3    | ✅ PASS | -1   | -1   | 找不到路线
4    | ✅ PASS | 3    | 3    | K足够大，选最便宜的迂回路线
5    | ✅ PASS | 6    | 6    | K不够，只能选中等价位路线
6    | ✅ PASS | 200  | 200  | 包含环，测试防重拦截机制
7    | ✅ PASS | -1   | -1   | 起点和终点根本不通
8    | ✅ PASS | 150  | 150  | 有多条同等转机次数的平行航线
9    | ✅ PASS | 50   | 50   | 复杂的梯子网络
10   | ✅ PASS | 0    | 0    | 极限: 单个节点不需要飞
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

Bellman-Ford 的底层物理意义就是：**外层循环执行了 $N$ 次，就代表探索了“最多经过 $N$ 条边”的最短路径。**
题目要求最多中转 $K$ 次，等价于**最多乘坐 $K + 1$ 趟航班（走 $K + 1$ 条边）**。
所以，我们只需要无脑让外层循环跑 $K + 1$ 次，跑完直接交卷！

**🔥 核心护城河：防止“串联爆炸”的克隆术**
如果在同一个循环里直接修改 `dist` 数组，会发生恐怖的“连环转机”：
比如 `A -> B` 刚更新完，紧接着遍历到 `B -> C`，它会立刻拿着 B 的新数据去更新 C。在这一趟循环里，信号瞬间跑了 2 条边！这直接破坏了“一次循环只能飞一趟”的限制。
**解法**：每次外层循环开始前，把 `dist` 复制一份叫 `clone_dist`。

- **看旧的**：从 `dist` 里读取历史价格。
- **写新的**：把算出来的新价格写进 `clone_dist` 里。
- 循环结束后：新老交替，`dist = clone_dist`。
完美斩断串联！
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(K \times E)$。外层循环 $K+1$ 次，内层遍历 $E$ 条边，极其稳定，没有任何多余的性能开销。
* **空间复杂度**：$O(V)$。只需要两个长度为 $V$ 的一维数组，甚至连图（邻接表）都不需要建，空间极度压缩！
:::

## 📝 题目 1514：概率最大的路径

::: {.callout-caution}
### 📖 题目描述
给你一个由 `n` 个节点（编号 `0` 到 `n - 1`）组成的**无向**图，该图由一个 `edges` 边缘列表组成，其中 `edges[i] = [a, b]` 表示节点 `a` 和 `b` 之间有一条边，其成功遍历的概率为 `succProb[i]` 。

请你找出从起点 `start_node` 到终点 `end_node` 的成功概率最大的路径，并返回其成功概率。
如果不存在从 `start_node` 到 `end_node` 的路径，请返回 `0` 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 3`, `edges = [[0,1],[1,2],[0,2]]`, `succProb = [0.5,0.5,0.2]`, `start = 0`, `end = 2`
    * **输出**：`0.25000`
    * **解释**：从 0 到 2 有两条路：
      直达：0 -> 2，概率为 0.2。
      中转：0 -> 1 -> 2，概率为 0.5 * 0.5 = 0.25。
      取最大值 0.25。

* **示例 2**：
    * **输入**：`n = 3`, `edges = [[0,1]]`, `succProb = [0.5]`, `start = 0`, `end = 2`
    * **输出**：`0.00000`
    * **解释**：节点 0 和 2 之间根本不连通，概率为 0。
:::

In [12]:
class Solution1514:
    def maxProbability_BellmanFord(self, n: int, edges: List[List[int]], succProb: List[float], start_node: int, end_node: int,) -> float:
        prob = [0.0] * n  # 初始概率全为 0，起点概率为 1
        prob[start_node] = 1.0
        for _ in range(n - 1):
            updated = False # 记录本轮是否有数据变动
            for i in range(len(edges)):
                if prob[edges[i][1]] * succProb[i] > prob[edges[i][0]]:
                    prob[edges[i][0]] = prob[edges[i][1]] * succProb[i]
                    updated = True
                if prob[edges[i][0]] * succProb[i] > prob[edges[i][1]]:  # 无向图必须反过来尝试用
                    prob[edges[i][1]] = prob[edges[i][0]] * succProb[i]
                    updated = True
            if not updated:  # 如果全图都没有更新，说明彻底收敛
                    break
        return prob[end_node]

    def maxProbability_SPFA(self, n: int, edges: List[List[int]], succProb: List[float], start_node: int, end_node: int,) -> float:
        adj = {i:[] for i in range(n)}  # 建图：无向图的邻接表
        for i in range(len(edges)):
            adj[edges[i][0]].append((succProb[i], edges[i][1]))
            adj[edges[i][1]].append((succProb[i], edges[i][0]))
        prob = [0.0] * n  # 初始化概率表
        prob[start_node] = 1.0
        queue = deque([start_node])  # SPFA 核心：双端队列 + 在队标志
        in_queue = [False] * n
        in_queue[start_node] = True # 起点已经在排队了
        while queue:
            curr_node = queue.popleft()
            in_queue[curr_node] = False
            for edge_p, next_node in adj[curr_node]:
                if prob[curr_node] * edge_p > prob[next_node]:
                    prob[next_node] = prob[curr_node] * edge_p
                    if not in_queue[next_node]:  # 如果它没在排队，把它塞进队伍里
                        in_queue[next_node] = True
                        queue.append(next_node)
        return prob[end_node]

In [14]:
#| code-fold: true
def test_1514(func1, func2):
    test_cases = [
        {"desc": "示例 1: 常规中转", "n": 3, "edges": [[0,1],[1,2],[0,2]], "succProb": [0.5,0.5,0.2], "start": 0, "end": 2, "expected": 0.25},
        {"desc": "示例 2: 根本不连通", "n": 3, "edges": [[0,1]], "succProb": [0.5], "start": 0, "end": 2, "expected": 0.0},
        {"desc": "示例 3: 直达最优", "n": 3, "edges": [[0,1],[1,2],[0,2]], "succProb": [0.5,0.5,0.3], "start": 0, "end": 2, "expected": 0.3},
        {"desc": "长链传递 (越来越小)", "n": 4, "edges": [[0,1],[1,2],[2,3]], "succProb": [0.8, 0.8, 0.8], "start": 0, "end": 3, "expected": 0.51200},
        {"desc": "星型辐射 (绕路反而大)", "n": 4, "edges": [[0,1],[1,2],[2,3],[0,3]], "succProb": [0.9, 0.9, 0.9, 0.1], "start": 0, "end": 3, "expected": 0.72900},
        {"desc": "极限: 单个节点无需移动", "n": 1, "edges": [], "succProb": [], "start": 0, "end": 0, "expected": 1.0},
        {"desc": "包含环，SPFA自动收敛测试", "n": 3, "edges": [[0,1],[1,2],[2,0]], "succProb": [0.5, 0.5, 0.5], "start": 0, "end": 1, "expected": 0.5},
        {"desc": "大跨度低概率", "n": 2, "edges": [[0,1]], "succProb": [0.00001], "start": 0, "end": 1, "expected": 0.00001},
        {"desc": "断层图", "n": 5, "edges": [[0,1],[1,2], [3,4]], "succProb": [0.5, 0.5, 0.8], "start": 0, "end": 4, "expected": 0.0},
        {"desc": "平行边博弈 (有多条绕远路)", "n": 4, "edges": [[0,1],[0,2],[1,3],[2,3]], "succProb": [0.4, 0.8, 0.4, 0.2], "start": 0, "end": 3, "expected": 0.16000} # 0-1-3 = 0.16; 0-2-3 = 0.16
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<8} | {'BellmanFord':<12} | {'SPFA':<8} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        # 考虑到浮点数精度，使用 round 保留 5 位小数进行比对
        actual_1 = round(func1(tc["n"], tc["edges"], tc["succProb"], tc["start"], tc["end"]), 5)
        actual_2 = round(func2(tc["n"], tc["edges"], tc["succProb"], tc["start"], tc["end"]), 5)
        expected = round(tc["expected"], 5)
        is_pass = actual_1 == actual_2 == expected
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {expected:<8} | {actual_1:<12} | {actual_2:<8} | {tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1514(Solution1514().maxProbability_BellmanFord,Solution1514().maxProbability_SPFA)

ID   | 结果     | 预期       | BellmanFord  | SPFA     | 描述
---------------------------------------------------------------------------
1    | ✅ PASS | 0.25     | 0.25         | 0.25     | 示例 1: 常规中转
2    | ✅ PASS | 0.0      | 0.0          | 0.0      | 示例 2: 根本不连通
3    | ✅ PASS | 0.3      | 0.3          | 0.3      | 示例 3: 直达最优
4    | ✅ PASS | 0.512    | 0.512        | 0.512    | 长链传递 (越来越小)
5    | ✅ PASS | 0.729    | 0.729        | 0.729    | 星型辐射 (绕路反而大)
6    | ✅ PASS | 1.0      | 1.0          | 1.0      | 极限: 单个节点无需移动
7    | ✅ PASS | 0.5      | 0.5          | 0.5      | 包含环，SPFA自动收敛测试
8    | ✅ PASS | 1e-05    | 1e-05        | 1e-05    | 大跨度低概率
9    | ✅ PASS | 0.0      | 0.0          | 0.0      | 断层图
10   | ✅ PASS | 0.16     | 0.16         | 0.16     | 平行边博弈 (有多条绕远路)
---------------------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题是图论中极其经典的“变种题”。它把传统的“求最小加法距离”变成了“求最大乘法概率”。
由于题目给定的概率都是 $P \le 1$，所以数字越乘越小，这与寻找“最短距离”的底层衰减逻辑完全同构！我们只需把 `min` 换成 `max`，把初始的无穷大换成 `0.0` 即可。

#### ⚔️ 方法一：Bellman-Ford 算法
**核心思想：无脑遍历，直到天下太平。**

1. **初始化**：大家初始概率都是 `0.0`，唯独起点是 `1.0` (100% 成功)。
2. **全局松弛**：图论第一定律，在一个包含 $N$ 个节点的图里，一条不绕圈的最优路径最多有 $N-1$ 条边。所以最外层循环跑 $N-1$ 次。
3. **无向图双向更新**：每次循环把所有边拿出来，因为是无向边，必须尝试用 `u` 更新 `v`，同时反过来尝试用 `v` 更新 `u`。
4. **提前下班 (极其关键的优化)**：如果在某一次全图扫荡中，没有任何人的概率发生改变（`updated == False`），说明最优解已经彻底收敛，直接 `break` 结束战斗，不用傻跑到 $N-1$ 次。

#### 🛡️ 方法二：SPFA 算法
**核心思想：谁的数据变好了，谁才有资格去通知邻居！**
Bellman-Ford 的痛点在于，即使某个节点的数据几百轮都没变过，它每一轮还要傻傻地去跟邻居比对。SPFA 用一个“双端队列”和“在队标志”完美解决了这个问题。

1. **建图与入队**：建立常规的邻接表 `adj`。起点带着 `1.0` 的概率入队，并在 `in_queue` 登记为 `True`。
2. **出队与解除封印**：每次从队首请出一位“数据刚变好”的节点 `curr_node`，立刻把它的 `in_queue` 状态改为 `False`。*(物理意义：解除封印，允许它未来再次变富有时重新排队)*。
3. **精准通知**：让这位节点去遍历自己的邻居。如果 `当前概率 * 路段概率 > 邻居现有的概率`，就果断更新邻居。
4. **新贵入队**：既然邻居的数据变好了，邻居也就有了去通知别人的资本！检查一下如果邻居此时不在队列里（`not in_queue[next_node]`），就赶紧把它塞进队列继续向外发散。
:::

::: {.callout-note}
### 💡 复杂度分析
* **Bellman-Ford**：
  - 时间复杂度：$O(V \times E)$。最坏情况下需要跑满 $V-1$ 轮全图扫描。
  - 空间复杂度：$O(V)$。连邻接表都不用建，极致省空间。
* **SPFA**：
  - 时间复杂度：平均情况下极其优秀，接近 $O(E)$。但在最坏情况（例如被特殊构造的稠密网格图“网暴”时）会退化为 $O(V \times E)$。
  - 空间复杂度：$O(V + E)$。需要建邻接表以及维护队列。在绝大多数日常实战中，SPFA 的速度碾压 Bellman-Ford。
:::

## 📝 题目 1334：阈值距离内邻居最少的城市

::: {.callout-caution}
### 📖 题目描述
有 `n` 个城市，按从 `0` 到 `n-1` 编号。给你一个边数组 `edges`，其中 `edges[i] =` [$from_i$, $to_i$, $weight_i$] 代表 $from_i$ 和 $to_i$ 两个城市之间有一条**双向**边，距离为 $weight_i$。
同时给你一个整数 `distanceThreshold`（距离阈值）。

一个城市的“可达邻居数”，是指那些**最短路径距离不超过 `distanceThreshold`** 的城市数量。

请你返回**可达邻居数最少**的城市。
**注意：**如果有多个城市的可达邻居数一样少，请返回**编号最大**的那个城市。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 4`, `edges = [[0,1,3],[1,2,1],[1,3,4],[2,3,1]]`, `distanceThreshold = 4`
    * **输出**：`3`
    * **解释**：
      城市 0 的可达城市有：1, 2 (共 2 个)
      城市 1 的可达城市有：0, 2, 3 (共 3 个)
      城市 2 的可达城市有：0, 1, 3 (共 3 个)
      城市 3 的可达城市有：1, 2 (共 2 个)
      城市 0 和 3 的可达城市一样少（都是 2 个）。题目要求返回编号最大的，所以返回 3。

* **示例 2**：
    * **输入**：`n = 5`, `edges = [[0,1,2],[0,4,8],[1,2,3],[1,4,2],[2,3,1],[3,4,1]]`, `distanceThreshold = 2`
    * **输出**：`0`
    * **解释**：
      城市 0 的可达城市只有自己（距离<=2内没有别人），共 0 个。（注：自身不算在邻居内）
      ...计算其他城市后，发现城市 0 的邻居最少，返回 0。
:::

In [28]:
class Solution1334:
    def findTheCity_BellmanFord(self, n: int, edges: List[List[int]], distanceThreshold: int) -> int:
        def BellmanFord(start: int) -> int:
            dist = [math.inf] * n
            dist[start] = 0
            for _ in range(n - 1):  # Bellman-Ford 核心：全图松弛 n-1 次
                updated = False
                for edge in edges:
                    if dist[edge[0]] + edge[2] < dist[edge[1]]:
                        dist[edge[1]] = dist[edge[0]] + edge[2]
                        updated = True
                    if dist[edge[1]] + edge[2] < dist[edge[0]]:
                        dist[edge[0]] = dist[edge[1]] + edge[2]
                        updated = True
                if not updated:  # 如果不更新提前结束
                    break
            reachable_count = -1  # 统计阈值内的城市数量（初始值为 -1 是为了扣除起点自己）
            for distance in dist:
                if distance <= distanceThreshold:
                    reachable_count += 1
            return reachable_count

        min_reachable = math.inf
        best_city = -1
        for curr_city in range(n):
            count = BellmanFord(curr_city)
            if count <= min_reachable:  # <= 极其关键：遇到平局时，让编号更大的后来者上位
                min_reachable = count
                best_city = curr_city
        return best_city

    def findTheCity_SPFA(self, n: int, edges: List[List[int]], distanceThreshold: int) -> int:
        adj = {i: [] for i in range(n)}  # 全场只建一次图
        for edge in edges:
            adj[edge[0]].append((edge[2], edge[1]))
            adj[edge[1]].append((edge[2], edge[0]))

        def SPFA(start: int) -> int:
            dist = [math.inf] * n
            dist[start] = 0
            queue = deque([start])  # SPFA 核心：双端队列 + 在队标志
            in_queue = [False] * n
            in_queue[start] = True  # 起点开始排队
            while queue:
                curr_node = queue.popleft()
                in_queue[curr_node] = False
                for weight, next_node in adj[curr_node]:
                    if dist[curr_node] + weight < dist[next_node]:
                        dist[next_node] = dist[curr_node] + weight
                        if not in_queue[next_node]:
                            queue.append(next_node)
                            in_queue[next_node] = True
            reachable_count = -1  # 统计阈值内的城市数量（初始值为 -1 是为了扣除起点自己）
            for distance in dist:
                if distance <= distanceThreshold:
                    reachable_count += 1
            return reachable_count

        min_reachable = math.inf
        best_city = -1
        for curr_city in range(n):
            count = SPFA(curr_city)
            if count <= min_reachable:  # <= 极其关键：遇到平局时，让编号更大的后来者上位
                min_reachable = count
                best_city = curr_city
        return best_city

In [29]:
#| code-fold: true
def test_1334(func1, func2):
    test_cases = [
        {"desc": "示例 1: 常规网络", "n": 4, "edges": [[0,1,3],[1,2,1],[1,3,4],[2,3,1]], "distanceThreshold": 4, "expected": 3},
        {"desc": "示例 2: 包含孤岛和长链", "n": 5, "edges": [[0,1,2],[0,4,8],[1,2,3],[1,4,2],[2,3,1],[3,4,1]], "distanceThreshold": 2, "expected": 0},
        {"desc": "全互不连通 (都为0个邻居，返回最大ID)", "n": 4, "edges": [], "distanceThreshold": 10, "expected": 3},
        {"desc": "阈值极大 (都能到达，返回最大ID)", "n": 3, "edges": [[0,1,1],[1,2,1]], "distanceThreshold": 100, "expected": 2},
        {"desc": "阈值极小 (谁也到不了谁，返回最大ID)", "n": 3, "edges": [[0,1,10],[1,2,10]], "distanceThreshold": 1, "expected": 2},
        {"desc": "星型网络 (中心点邻居多，边缘点少)", "n": 4, "edges": [[0,1,1],[0,2,1],[0,3,1]], "distanceThreshold": 1, "expected": 3}, # 1, 2, 3 都只有 1 个邻居 (0号)，取最大的 3
        {"desc": "直线形网络 (边界点邻居最少)", "n": 5, "edges": [[0,1,1],[1,2,1],[2,3,1],[3,4,1]], "distanceThreshold": 2, "expected": 4},
        {"desc": "绕远路比直达更近 (SPFA 必测)", "n": 3, "edges": [[0,1,10],[0,2,1],[1,2,1]], "distanceThreshold": 3, "expected": 2}, # 0->2->1 距离2. 所有点互相距离<=3，都连通，选最大 2
        {"desc": "两个完全相同的独立网络 (返回后面那个)", "n": 4, "edges": [[0,1,1],[2,3,1]], "distanceThreshold": 2, "expected": 3},
        {"desc": "刚好卡在阈值线上", "n": 3, "edges": [[0,1,5],[1,2,5]], "distanceThreshold": 5, "expected": 2}, # 0邻居[1], 1邻居[0,2], 2邻居[1]. 0和2最少，返回2
        {"desc": "极限: 单个城市", "n": 1, "edges": [], "distanceThreshold": 1, "expected": 0}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'BellmanFord':<12} | {'SPFA':<8} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        actual_1 = func1(tc["n"], tc["edges"], tc["distanceThreshold"])
        actual_2 = func2(tc["n"], tc["edges"], tc["distanceThreshold"])
        is_pass = actual_1 == actual_2 == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual_1:<12} | {actual_2:<8} |{tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1334(Solution1334().findTheCity_BellmanFord, Solution1334().findTheCity_SPFA)

ID   | 结果     | 预期   | BellmanFord  | SPFA     | 描述
---------------------------------------------------------------------------
1    | ✅ PASS | 3    | 3            | 3        |示例 1: 常规网络
2    | ✅ PASS | 0    | 0            | 0        |示例 2: 包含孤岛和长链
3    | ✅ PASS | 3    | 3            | 3        |全互不连通 (都为0个邻居，返回最大ID)
4    | ✅ PASS | 2    | 2            | 2        |阈值极大 (都能到达，返回最大ID)
5    | ✅ PASS | 2    | 2            | 2        |阈值极小 (谁也到不了谁，返回最大ID)
6    | ✅ PASS | 3    | 3            | 3        |星型网络 (中心点邻居多，边缘点少)
7    | ✅ PASS | 4    | 4            | 4        |直线形网络 (边界点邻居最少)
8    | ✅ PASS | 2    | 2            | 2        |绕远路比直达更近 (SPFA 必测)
9    | ✅ PASS | 3    | 3            | 3        |两个完全相同的独立网络 (返回后面那个)
10   | ✅ PASS | 2    | 2            | 2        |刚好卡在阈值线上
11   | ✅ PASS | 0    | 0            | 0        |极限: 单个城市
---------------------------------------------------------------------------
测试总结: 通过 11/11


::: {.callout-important}
### 💡 思路讲解

这道题的核心痛点在于：我们不再是只求“从城市 A 出发”的距离，而是需要**上帝视角，求出所有城市到所有城市的距离**（多源最短路径问题）。

在我们还没掌握终极神技 Floyd 之前，我们可以用手里的**单源最短路引擎（SPFA 或 Bellman-Ford）**进行“降维打击”：

**核心三步走：**
1. **外层大循环 (上帝视角)**：写一个 `for i in range(n)` 的循环，把每一个城市 `i` 轮流当做“起点”。
2. **内层跑引擎 (降维计算)**：在循环内部，启动你写好的 `SPFA(i)`。引擎跑完后，你会得到一个 `dist` 数组，里面装满了全图其他城市到城市 `i` 的最短距离。
3. **统计与选拔 (极其关键的逻辑)**：
   - 遍历 `dist` 数组，统计距离 $\le$ `distanceThreshold` 的城市数量 `count`（注意：自己到自己的距离是 0，也被算进去了，但这不影响最终比大小的相对结果，或者像你的代码里初始化 `count = -1` 巧妙排除了自己）。
   - **平局选拔机制**：题目要求邻居数量一样少时，取**编号最大**的城市。因为我们的外层循环是从 `0` 走到 `n-1`，所以只要当前城市的 `count <= 历史 min_val`，就果断更新答案！这个 `=` 号完美保证了“后来者居上”。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：
  - **套 Bellman-Ford**：$O(V \times V \times E) = O(V^2 E)$。外层循环 $V$ 次，内层 Bellman 跑 $V \times E$。
  - **套 SPFA**：平均 $O(V \times E)$，最坏退化到 $O(V^2 E)$。极其适合稀疏图。
* **空间复杂度**：$O(V + E)$。建一次图，每次循环内部开辟一个大小为 $V$ 的队列和距离数组。
:::